# mailman - training the extraction model

Trains a token classifier that labels each word of an invoice with the field it belongs to,
then exports it for the mailman pipeline to serve locally. No API key, no per-call cost.

**Run this on Colab with a GPU.** Runtime -> Change runtime type -> T4 GPU. It is free, and
a run takes a few minutes.

## Why a tagger and not a generative model

The pipeline already has the document's text layer, and every field wanted is a span that
physically appears on the page. A tagger returns *where* it found each value, so a value it
returns came from the document. That removes a whole class of failure - a tagger cannot
invent an invoice number that was never printed. It can only mislabel one that was.

The cost is that it cannot infer anything not written down, and it has no idea what an
invoice *means*. Those are real limits and they belong in the README.

## What this produces

A directory of weights (~250 MB) that `mailman/trained.py` loads. The weights do **not** go
in git - too large - and will not fit a free hosting tier's memory. The heuristic extractor
is what deploys; this is the local and showcase path, and the interesting number is the gap
between the two.


In [ ]:
# What GPU did Colab give us, if any.
import shutil
import subprocess

# shutil.which first. On a CPU runtime nvidia-smi does not exist, and subprocess.run
# raises FileNotFoundError rather than returning empty output - so the `or ...` fallback
# never got the chance to run, and the very first cell of the notebook blew up.
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No GPU on this runtime.")
    print("That is fine - the run-size cell below sizes the job for CPU automatically.")
    print("For a GPU: Runtime -> Change runtime type -> T4 GPU, subject to Colab quota.")

In [ ]:
# torch is NOT upgraded here, on purpose.
#
# Colab ships torch, torchvision and torchaudio built against each other. Upgrading
# torch alone leaves torchvision bound to a version that no longer exists, and its
# operators fail to register:
#
#     RuntimeError: operator torchvision::nms does not exist
#
# transformers imports torchvision lazily, so the failure surfaces later as an
# unrelated-looking 'Could not import module Trainer'. The torch Colab gives you
# already has CUDA and is the one to use.
%pip install -q -U transformers datasets accelerate seqeval
print()
print("installed - now run the version check below")

### Check the install before training

Three seconds here saves a confusing failure much later. If torch and torchvision
disagree, the first symptom is a `Trainer` import error dozens of cells away, which
points at the wrong thing entirely.

In [ ]:
# Run this before training. It is three seconds and it turns a confusing failure
# forty cells later into an obvious one here.
import torch
import transformers

print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")

try:
    import torchvision
    print(f"torchvision  {torchvision.__version__}")
    torchvision.ops.nms(
        torch.tensor([[0.0, 0.0, 1.0, 1.0]]), torch.tensor([0.5]), 0.5
    )
    print("torchvision ops load correctly")
except Exception as exc:
    print()
    print(f"torchvision is broken: {type(exc).__name__}: {exc}")
    print("torch and torchvision disagree. Runtime -> Restart session, then run the")
    print("cells again WITHOUT upgrading torch. Do not pip install torch here.")
    raise SystemExit(1)

print()
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU. It will train on CPU,")
    print("slowly, if you would rather not.")

### Run size

**This run is a controlled comparison. It needs a GPU, and therefore 6 epochs.**

The question it answers is whether the label vocabulary matters, now that four of the eight
lists are actually wired into the generator for the first time. The run to compare against is
**run 3: 6 epochs, large vocabulary, shifted 47.4%, gap 52.6%**. Same seed, same epochs, only
the emitted vocabulary differing - one variable.

On CPU the cell below drops to 2 epochs, and **the comparison is then worthless**, because two
things will have moved. That is not hypothetical: run 2 was 2 epochs on a new vocabulary
against run 1's 6 epochs on the old one, both variables moved, and the +5.7 points it reported
could not be attributed to either. The CPU fallback is what made the epoch count feel like an
environmental detail rather than an experimental variable. The cell prints a warning if it
happens.

An earlier version of this note said the epoch count was settled - F1 hit 1.000 at epoch 1, so
epochs 2 to 6 "changed nothing measurable". **That was wrong, and it was the in-distribution
F1 saying it.** Run 3 against run 2, same data and vocabulary, differing only in epochs:
`VENDOR_NAME` 27% to 100%, `LINE_AMOUNT` 32% to 72%, `TOTAL` 44% down to 6%, overall +1 point.
Training longer redistributes which fields it gets right. A metric that is saturated at 1.000
cannot see any of that.

Sequence length is deliberately left alone. Measured over 800 generated invoices the median is
132 word-pieces and the longest is 197, so nothing reaches the 512 cap - and the collator pads
to the longest sequence in each batch rather than to the cap. Lowering `max_length` looks like
an obvious saving and is not one. Note the mismatch it hides, though: training documents carry
one to six line items, while the evaluation corpus has one with forty, and the totals block
sits last and would be the first thing truncated.


In [ ]:
import torch

ON_GPU = torch.cuda.is_available()

# The run being compared against. Kept here so the comparison is stated before the run
# rather than reconstructed after it. UPDATE THIS after every run, or the next run silently
# measures itself against a stale target and overstates its own improvement.
#
#   run 1   6 epochs, small vocabulary                    shifted 40.7%   gap 59.3%
#   run 2   2 epochs, vocabulary half-applied             shifted 46.4%   gap 53.6%
#   run 3   6 epochs, vocabulary half-applied             shifted 47.4%   gap 52.6%
#   run 4   6 epochs, vocabulary fully applied            shifted 70.7%   gap 29.3%
#   run 5   6 epochs, + identifiers, structure, descriptions  shifted 84.1%   gap 15.9%
#   run 6   this one
BASELINE = {"run": 5, "epochs": 6, "shifted": 0.841, "gap": 0.159}

if ON_GPU:
    TRAINING_DOCUMENTS = 4000
    EPOCHS = 6
    BATCH_SIZE = 16
    EVAL_DOCUMENTS = 200
else:
    # Two epochs finishes on CPU in minutes. It also breaks the comparison - see the warning
    # below. The reduction is in epochs rather than in documents, because the vocabulary is
    # the thing the model has to generalise over and cutting documents would defeat the point
    # of having enlarged it.
    TRAINING_DOCUMENTS = 4000
    EPOCHS = 2
    BATCH_SIZE = 8
    EVAL_DOCUMENTS = 100

print(f"device            {'GPU' if ON_GPU else 'CPU'}")
print(f"training docs     {TRAINING_DOCUMENTS}")
print(f"epochs            {EPOCHS}")
print(f"batch size        {BATCH_SIZE}")
print(f"eval documents    {EVAL_DOCUMENTS}  (per set, for the serving-path scores)")
print()
print(f"comparing against run {BASELINE['run']}: {BASELINE['epochs']} epochs, "
      f"shifted {BASELINE['shifted']:.1%}, gap {BASELINE['gap']:.1%}")

if EPOCHS == BASELINE["epochs"]:
    print("epochs match the baseline. What differs is the generator configuration")
    print("printed by the data cell below - keep that in mind when attributing the result.")
else:
    print()
    print("*" * 70)
    print(f"WARNING: {EPOCHS} epochs against the baseline's {BASELINE['epochs']}.")
    print("Two variables will have moved - vocabulary AND training time - and the result")
    print("cannot be attributed to either. This is exactly the mistake run 2 made.")
    print()
    print("Get a GPU (Runtime -> Change runtime type -> T4) and run it again, or set")
    print("EPOCHS = 6 by hand and accept that it will take a few hours on CPU.")
    print("Either way, do not record the number as a comparison against run 3.")
    print("*" * 70)

if not ON_GPU:
    print()
    print("No GPU: roughly 15-30 minutes at 2 epochs, and hours at 6.")
    print("Keep the browser tab open - an idle Colab CPU session disconnects.")


## 1. Training data

The invoices are generated, which means **the labels come free**. The generator knows what
it printed and where, so every training example is labelled by construction rather than by
hand. This is the same reason the project's evaluation corpus is safe to build after the
pipeline rather than before it.

That is also this notebook's biggest weakness, and it should be said plainly: a model
trained only on invoices from one generator learns that generator. The layouts below vary on
purpose - different label wording, different orderings, different date and money formats -
but it is still one author's idea of what an invoice looks like. Real or public samples
mixed in are what would make the numbers mean something, and the place to do that is the
cell marked **Optional** further down.


In [ ]:
import random
from datetime import date, timedelta

FIELD_LABELS = [
    "INVOICE_NUMBER", "VENDOR_NAME", "BUYER_NAME", "ISSUE_DATE", "DUE_DATE",
    "CURRENCY", "SUBTOTAL", "TAX", "TOTAL",
    "LINE_DESCRIPTION", "LINE_QUANTITY", "LINE_UNIT_PRICE", "LINE_AMOUNT",
]
LABELS = ["O"] + [f"{p}-{f}" for f in FIELD_LABELS for p in ("B", "I")]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}

# Deliberately large. Run 1 scored 100% in-distribution and 40.7% shifted, and the per-field
# breakdown showed why: the fields keyed to a label word collapsed to 0% when the wording
# changed, while the fields identified by position stayed above 94%.
#
# Run 4 settled what that was worth. With all of these actually wired into the generator -
# four of them were not, for three runs - the shifted score went 47.4% to 70.7% and the gap
# halved. Four phrasings for a total is a lookup table; thirteen is a pattern.
VENDORS = [
    "Acme Corp Ltd", "Northgate Supplies", "Bluewater Logistics", "Harrow & Finch",
    "Tessellate Systems", "Meridian Print Works", "Corvid Engineering", "Ashland Paper Co",
    "Peregrine Tooling", "Halvorsen Freight", "Studio Ninefold", "Brackenridge Timber",
    "Quill & Bassett", "Ironhill Fabrication", "Marlowe Scientific", "Devereux Catering",
    "Sandpiper Cleaning", "Kettering Glassworks", "Vermeer Data Services", "Oakfield Plant Hire",
]
BUYERS = [
    "Kestrel Retail Ltd", "Orchard Foods", "Vantage Media", "Pelham Group",
    "Thornbury Hotels", "Lansdowne Clinics", "Ridgeway Motors", "Calloway Interiors",
    "Fenwick Property", "Alderton Schools Trust", "Bexley Wholesale", "Maplewood Care",
]
GOODS = [
    "Widget assembly", "Freight charge", "Consulting hours", "Packaging", "Installation",
    "Annual licence", "Courier", "Site survey", "Scaffold hire", "Calibration",
    "Archive retrieval", "Translation", "Emergency callout", "Waste removal",
    "Server rental", "Design retainer", "Paint, matt white", "Cable, CAT6 per metre",
    "Safety inspection", "Bulk print run", "Legal review", "Equipment servicing",
]
CURRENCIES = [("GBP", "GBP "), ("USD", "$"), ("EUR", "EUR "), ("GBP", "£"),
              ("EUR", "€"), ("CAD", "CAD "), ("AUD", "AUD ")]

NUMBER_LABELS = [
    "Invoice Number:", "Invoice No.", "Invoice #", "INV NUMBER", "Our reference",
    "Document ID", "Bill No", "Reference:", "Ref", "Invoice ref", "No.",
    "Account document", "Statement number", "Tax invoice no", "Doc ref",
]
DATE_LABELS = [
    "Invoice Date:", "Date of Issue:", "Date:", "Issued:", "Raised on", "Tax point",
    "Statement date", "Dated", "Invoice dated", "Issue date", "Billing date",
]
DUE_LABELS = [
    "Due Date:", "Payment Due:", "Due:", "Settlement by", "Payable before",
    "Payment due by", "Terms - due", "Pay by", "Remit by",
]
SUBTOTAL_LABELS = [
    "Subtotal", "Sub total", "Net", "Net total", "Goods value", "Goods total",
    "Total excl. tax", "Total net", "Amount before tax", "Nett",
]
TAX_LABELS = [
    "VAT", "Tax", "Sales Tax", "Duty", "Output tax", "VAT @ 20%", "GST",
    "Tax charged", "VAT amount",
]
TOTAL_LABELS = [
    "Total Due", "Grand Total", "TOTAL", "Amount Due", "Balance now due", "To pay",
    "NET PAYABLE", "Amount payable", "Total payable", "Balance due", "Invoice total",
    "Total incl. tax", "Please pay",
]
TABLE_HEADERS = [
    "Description Qty Unit Price Amount", "Particulars Units Rate Value",
    "Item Quantity Price Total", "Details Qty Rate Net", "Service Units Cost Line total",
]
CURRENCY_LABELS = [
    "Currency", "Priced in", "Amounts in", "Settlement currency", "All values in",
]

# --- the constants run 4 exposed, one layer down --------------------------------------
#
# Run 4 fixed the constant *labels* and the fields keyed to them went from 0% to ~73%. It
# left INVOICE_NUMBER at 13.0% and ISSUE_DATE at 12.5% - within half a point of each other,
# which is the span-merging signature, and the two of them were recorded merging into one
# span as far back as run 2.
#
# The cause is the same bug with the *values* as the constant. For four runs the generator
# emitted exactly one invoice-number shape:
#
#     f"INV-{issued.year}-{rng.randrange(1000, 9999)}"
#
# so INVOICE_NUMBER was learnable as "the token beginning INV-". The shifted set writes
# 123/2026/45, which shares nothing with that and is additionally date-shaped, so it merges
# into the issue date behind it. The real corpus already says the constant is wrong -
# INV-2026-0042, NS-88213, BW-2026-771, MPW-3310, AP-2026-5120: at least three shapes across
# eleven documents, with the prefix varying by vendor.
NUMBER_PREFIXES = [
    "INV", "NS", "BW", "CN", "TS", "MPW", "CE", "AP", "PT", "SS", "AC", "GH",
    "FR", "QB", "IF", "MS", "DC", "SC", "KG", "VD", "OP", "TX", "REF", "DOC", "BIL",
]


def an_invoice_number(rng, year, vary=True):
    """One invoice number. Several real shapes, none of them the shifted set's.

    The shifted generator uses `\\d{3}/\\d{4}/\\d{2}`; nothing here produces that, so the
    held-out set stays held out. What the model has to learn instead of a prefix is "the
    thing that comes after the number label", which is the only rule that survives.
    """
    if not vary:
        return f"INV-{year}-{rng.randrange(1000, 9999)}"

    prefix = rng.choice(NUMBER_PREFIXES)
    serial = rng.randrange(1, 99999)
    shape = rng.randrange(7)
    if shape == 0:
        return f"{prefix}-{year}-{serial % 10000:04d}"
    if shape == 1:
        return f"{prefix}-{serial}"
    if shape == 2:
        return f"{prefix}{serial % 10000:04d}"
    if shape == 3:
        return f"{year}/{serial % 10000:04d}"
    if shape == 4:
        return f"{prefix}/{year}/{serial % 1000:03d}"
    if shape == 5:
        return f"{serial}"
    return f"{prefix}-{serial % 1000:03d}-{year}"


# Text that carries no field at all, labelled O. Its job is to stop "the next token after
# anything" being a reliable cue, and to put identifier-shaped and money-shaped strings on
# the page that are NOT the invoice number and NOT a total. `PO-4471` is there on purpose.
FILLER_LINES = [
    "Payment terms 30 days net", "VAT Reg No GB 123 4567 89", "Page 1 of 1",
    "Remittance advice required", "Please quote our reference on payment",
    "Registered in England and Wales", "E&OE", "Thank you for your business",
    "PO Number PO-4471", "Delivery note enclosed", "Order ref ORD-99120",
    "All goods remain our property until paid in full", "Sort code 20-00-00",
    "Queries to the accounts department", "This is not a receipt",
    "Goods supplied under our standard conditions", "Ref your order dated last month",
]

TITLES = [
    "INVOICE", "TAX INVOICE", "SALES INVOICE", "INVOICE / STATEMENT", "VAT INVOICE",
]
BUYER_LABELS = [
    "Bill To:", "Invoice To:", "Customer", "Sold To:", "Client", "Buyer", "Charge to",
]
STREETS = [
    "Fleet Street", "Mill Road", "Kings Way", "Harbour Approach", "Cadogan Place",
    "Tannery Lane", "Bridgegate", "West Quay", "Sowerby Rise", "Camberwell Green",
]

# Line descriptions were 34.0% shifted after run 4. Twenty-two fixed strings is a lookup
# table for an open-vocabulary field; composing them gives a few thousand, so the model has
# to learn the description by its position in the row rather than by recognising it.
GOODS_PREFIXES = ["", "", "", "Premium ", "Standard ", "Bulk ", "Refurbished ", "Express "]
GOODS_SUFFIXES = ["", "", "", " - per unit", " (batch)", ", grade A", " - monthly",
                  " per metre", " - second fix", " (backordered)"]


def money(value, symbol):
    return f"{symbol}{value:,.2f}"


# The number of formats `a_date` knows. Named, because the training generator drew from
# `rng.randrange(4)` while the shifted generator drew from `rng.randrange(6)` - so two of the
# six formats appeared in a third of the shifted documents and in no training document at
# all, and ISSUE_DATE's ~14% was partly guaranteed rather than learned.
DATE_STYLES = 6


def a_date(d, style):
    if style == 0:
        return d.isoformat()
    if style == 1:
        return d.strftime("%d/%m/%Y")
    if style == 2:
        return d.strftime("%d %B %Y")
    if style == 3:
        return d.strftime("%b %d, %Y")
    if style == 4:
        return d.strftime("%d-%b-%Y")
    return d.strftime("%d.%m.%Y")


In [ ]:
from collections import Counter, defaultdict

# What varies in the generated documents. Every flag is a separate experiment, and they are
# separate flags rather than one switch so the next run can attribute its result.
#
#   vary_labels        the run 4 fix. Four label lists were defined and never drawn from for
#                      three runs; wiring them in took shifted 47.4% -> 70.7%, gap halved.
#   vary_identifiers   NEW. The invoice number was ONE shape - INV-{year}-{4 digits} - in
#                      every document of every run so far. INVOICE_NUMBER 13.0%.
#   vary_structure     NEW, and the lever nothing has tested. Field order was identical in
#                      every training document, so position was a perfect cue in training
#                      and none at all on anything else.
#   vary_descriptions  NEW. Twenty-two fixed strings for an open-vocabulary field.
#                      LINE_DESCRIPTION 34.0%.
#
# To attribute a result, turn one off and rerun: everything else is seeded identically.
GENERATOR = {
    "vary_labels": True,
    "vary_identifiers": True,
    "vary_structure": True,
    "vary_descriptions": True,
}

# The vocabulary the generator is supposed to draw from, by name.
#
# SUBTOTAL_LABELS, TAX_LABELS, TABLE_HEADERS and CURRENCY_LABELS were written into the cell
# above and never read by generate_invoice: every training document said "Subtotal",
# "Currency" and "Description Qty Unit Price Amount", and the tax label came from a hardcoded
# list of three. SUBTOTAL and TAX sat at 0% on the shifted set because of it, while TOTAL -
# the one totals-block list that was wired in - was the only one that moved.
VOCABULARY = {
    "VENDORS": VENDORS, "BUYERS": BUYERS, "GOODS": GOODS,
    "NUMBER_LABELS": NUMBER_LABELS, "DATE_LABELS": DATE_LABELS, "DUE_LABELS": DUE_LABELS,
    "SUBTOTAL_LABELS": SUBTOTAL_LABELS, "TAX_LABELS": TAX_LABELS,
    "TOTAL_LABELS": TOTAL_LABELS, "TABLE_HEADERS": TABLE_HEADERS,
    "CURRENCY_LABELS": CURRENCY_LABELS, "TITLES": TITLES,
    "BUYER_LABELS": BUYER_LABELS, "FILLER_LINES": FILLER_LINES, "STREETS": STREETS,
}

DRAWS = Counter()
DRAWN_PHRASES = defaultdict(set)


def pick(rng, name):
    """Draw from a named vocabulary list, recording that the list was drawn from.

    The draw is recorded here rather than inferred from the generated text, because two
    earlier versions of the guard below inspected the text and both were fooled by the very
    bug they were written to catch. The hardcoded constants were `"Subtotal"` and
    `"Currency"` - members of their own lists - so "did any phrase from this list appear"
    was true. Raising the bar to "did two appear" still passed, because the hardcoded tax
    list was three members of `TAX_LABELS`, and because `"Net"` turns up inside the table
    header `"Details Qty Rate Net"`.

    A check that reads the output can be satisfied by a coincidence. A check on the call
    cannot.
    """
    choices = VOCABULARY[name]
    if not GENERATOR["vary_labels"] and name in ("SUBTOTAL_LABELS", "TAX_LABELS",
                                                 "TABLE_HEADERS", "CURRENCY_LABELS"):
        choices = choices[:1]          # the pre-run-4 behaviour, for the ablation
    phrase = rng.choice(choices)
    DRAWS[name] += 1
    DRAWN_PHRASES[name].add(phrase)
    return phrase


def a_description(rng):
    base = pick(rng, "GOODS")
    if not GENERATOR["vary_descriptions"]:
        return base
    return f"{rng.choice(GOODS_PREFIXES)}{base}{rng.choice(GOODS_SUFFIXES)}"


def generate_invoice(rng):
    """Return (words, labels) for one invoice.

    Labels are attached as the text is written, not matched afterwards. Matching a value
    back to the text is where silent labelling bugs live - a total of 25.00 that also
    appears as a line amount would tag both.

    The document is built out of blocks rather than as one straight run of `emit` calls, so
    the order of the header blocks can vary. That is the point: with a fixed order, position
    alone identifies every field and the model has no reason to read the labels.
    """
    words, labels = [], []

    def emit(text, label=None):
        parts = str(text).split()
        for i, part in enumerate(parts):
            words.append(part)
            if label is None:
                labels.append("O")
            else:
                labels.append(("B-" if i == 0 else "I-") + label)

    structured = GENERATOR["vary_structure"]

    code_, symbol = rng.choice(CURRENCIES)
    date_style = rng.randrange(DATE_STYLES)
    issued = date(2026, 1, 1) + timedelta(days=rng.randrange(360))
    due = issued + timedelta(days=rng.choice([14, 30, 45, 60]))
    number = an_invoice_number(rng, issued.year, vary=GENERATOR["vary_identifiers"])

    def block_number():
        emit(pick(rng, "NUMBER_LABELS"))
        emit(number, "INVOICE_NUMBER")

    def block_issue():
        emit(pick(rng, "DATE_LABELS"))
        emit(a_date(issued, date_style), "ISSUE_DATE")

    def block_due():
        emit(pick(rng, "DUE_LABELS"))
        emit(a_date(due, date_style), "DUE_DATE")

    def block_buyer():
        emit(pick(rng, "BUYER_LABELS") if structured else "Bill To:")
        emit(pick(rng, "BUYERS"), "BUYER_NAME")

    def block_currency():
        emit(pick(rng, "CURRENCY_LABELS"))
        emit(code_, "CURRENCY")

    # Header. The vendor stays first: it is first on essentially every real invoice, and
    # `_vendor_name` in the heuristic depends on that too.
    emit(pick(rng, "VENDORS"), "VENDOR_NAME")
    street = pick(rng, "STREETS") if structured else "Mill Road"
    emit(f"{rng.randrange(1, 200)} {street}")
    if not structured:
        emit("INVOICE")
    elif rng.random() < 0.85:
        emit(pick(rng, "TITLES"))

    # The metadata blocks, in an order that is fixed only when vary_structure is off.
    meta = [block_number, block_issue]
    if rng.random() < 0.85:
        meta.append(block_due)
    meta.append(block_buyer)
    currency_at_end = True
    if structured:
        rng.shuffle(meta)
        if rng.random() < 0.4:            # the currency note sometimes sits in the header
            meta.append(block_currency)
            currency_at_end = False

    for block in meta:
        block()
        if structured and rng.random() < 0.25:
            emit(pick(rng, "FILLER_LINES"))

    # The table. Rows always follow their header, and the totals always follow the rows -
    # that ordering is real and the arithmetic depends on it.
    emit(pick(rng, "TABLE_HEADERS"))

    subtotal = 0.0
    for _ in range(rng.randrange(1, 7)):
        qty = rng.randrange(1, 30)
        unit = round(rng.uniform(4, 900), 2)
        amount = round(qty * unit, 2)
        subtotal += amount
        emit(a_description(rng), "LINE_DESCRIPTION")
        emit(str(qty), "LINE_QUANTITY")
        emit(money(unit, symbol), "LINE_UNIT_PRICE")
        emit(money(amount, symbol), "LINE_AMOUNT")

    subtotal = round(subtotal, 2)
    tax = round(subtotal * rng.choice([0.0, 0.05, 0.2]), 2)
    total = round(subtotal + tax, 2)

    # A field that is always present is a field whose absence has never been seen. Real
    # invoices skip the subtotal line when there is no tax, and skip the tax line entirely.
    show_subtotal = True if not structured else rng.random() < 0.9
    show_tax = True if not structured else rng.random() < 0.85

    if show_subtotal:
        emit(pick(rng, "SUBTOTAL_LABELS"))
        emit(money(subtotal, symbol), "SUBTOTAL")
    if show_tax:
        emit(pick(rng, "TAX_LABELS"))
        emit(money(tax, symbol), "TAX")
    emit(pick(rng, "TOTAL_LABELS"))
    emit(money(total, symbol), "TOTAL")

    if currency_at_end:
        block_currency()
    if structured and rng.random() < 0.3:
        emit(pick(rng, "FILLER_LINES"))

    return words, labels


rng = random.Random(20260901)   # fixed, so a rerun trains on the same data
examples = [generate_invoice(rng) for _ in range(TRAINING_DOCUMENTS)]
print(f"{len(examples)} invoices")
print("generator:", ", ".join(f"{k}={v}" for k, v in GENERATOR.items()))
print()
print("first 30 tokens of one:")
w, l = examples[0]
for word, label in list(zip(w, l))[:30]:
    print(f"  {word:24} {label}")


# A list defined and never drawn from is silent: the notebook runs, the manifest reports
# numbers, and the numbers describe the old distribution. So it is asserted, not hoped for.
# Only the lists the current configuration is supposed to draw from. With vary_structure
# off there is no title, no filler, one street and one buyer label - that is the pre-run-5
# generator, and expecting those lists to be drawn would make the ablation impossible to run.
_expected = set(VOCABULARY)
if not GENERATOR["vary_structure"]:
    _expected -= {"TITLES", "BUYER_LABELS", "FILLER_LINES", "STREETS"}
_never_drawn = sorted(_expected - set(DRAWS))
assert not _never_drawn, (
    f"vocabulary defined but never drawn from by generate_invoice: {_never_drawn}. "
    "A list that is not wired in makes the run measure the old distribution while the "
    "notebook reports it as the new one."
)

# Every date format reachable, for the same reason: the training generator drew from
# randrange(4) and the shifted set from randrange(6), so two of the six formats appeared in
# a third of the shifted documents and in no training document at all.
_dates_seen = {a_date(date(2026, 8, 14), s) for s in range(DATE_STYLES)}
assert len(_dates_seen) == DATE_STYLES, f"date styles collide: {sorted(_dates_seen)}"

# Field order really does vary now, rather than being intended to. Compares the sequence of
# field labels between documents; with a fixed order every document gives the same sequence.
_orders = {tuple(dict.fromkeys(t[2:] for t in labs if t != "O")) for _, labs in examples[:500]}
if GENERATOR["vary_structure"]:
    assert len(_orders) > 5, (
        f"vary_structure is on but only {len(_orders)} field orderings in 500 documents - "
        "the shuffle is not reaching the emitted text."
    )
print()
print(f"distinct field orderings in 500 documents: {len(_orders)}")

_numbers = {w for ws, labs in examples[:500] for w, t in zip(ws, labs) if t == "B-INVOICE_NUMBER"}
_shapes = {"".join("9" if c.isdigit() else ("A" if c.isalpha() else c) for c in n) for n in _numbers}
print(f"distinct invoice-number shapes in 500 documents: {len(_shapes)}")
# Per description, not per document - joining every description in a document counts
# documents, which would have looked like variety that is not there.
_descriptions, _current = set(), []
for _ws, _labs in examples[:500]:
    for _w, _t in zip(_ws, _labs):
        if _t == "B-LINE_DESCRIPTION":
            if _current:
                _descriptions.add(" ".join(_current))
            _current = [_w]
        elif _t == "I-LINE_DESCRIPTION":
            _current.append(_w)
        elif _current:
            _descriptions.add(" ".join(_current))
            _current = []
    if _current:
        _descriptions.add(" ".join(_current))
        _current = []
print(f"distinct line descriptions in 500 documents: {len(_descriptions)}")

print()
print(f"vocabulary: {len(VOCABULARY)} lists, every one drawn from")
for name in sorted(VOCABULARY):
    seen, size = len(DRAWN_PHRASES[name]), len(VOCABULARY[name])
    note = "" if seen == size else f"  ({size - seen} not drawn in this sample)"
    print(f"  {name:18} {seen}/{size} phrases, {DRAWS[name]} draws{note}")
print(f"date formats: {DATE_STYLES}, all reachable")


## 1b. Real invoices from Kaggle - THIS ROUTE IS CLOSED

**Leave `USE_KAGGLE = False`.** This section is kept because the dead end is worth recording,
not because it works.

The set was **High-Quality Invoice Images for OCR** by Osama Hosam Abdellatif. The claim that
sent us here - "1,489 fully annotated samples with structured JSON metadata and raw OCR text"
- came from the Voxel51 HuggingFace card describing the *FiftyOne dataset they built*, not the
Kaggle artifact. The download was run. It contains:

    0 json files, 8181 jpgs, 0 txt

No annotations of any kind. A tagger cannot train on unlabelled pictures, and the labels exist
only inside the FiftyOne copy - the same copy whose Parquet conversion dropped them. The
licence printed at download time is **DbCL-1.0**, not the ODbL recorded earlier.

Turning this on costs a gigabyte of images, a Kaggle token upload, and nothing else. The
converter and inspector cells below are left intact and are correct; they simply have nothing
to read.

**That is the third time this project was misled by a description rather than the artifact** -
the HuggingFace mirror's schema, the export's file list, and this. The inspector cell was part
of the problem: it counted `.json`, `.jpg` and `.txt` and reported "0 json files" while saying
nothing about the 8,181 other files. It now counts every extension present and says plainly
when a download carries no labels at all.

**The remaining candidates for real documents**, neither yet attempted:

| Set | Terms | Size |
| --- | --- | --- |
| CORD | receipts, CC-BY-4.0 | 800 train |
| RealKIE FCC Invoices | real invoices, CC-BY-NC, direct download, no form | 370 |

DocILE was rejected: its terms forbid third-party access, which kills the hosted-demo goal.


In [ ]:
USE_KAGGLE = False   # the Kaggle set has no annotations - see the cell above. Leave it False.

# Magics stay at the top level rather than inside the `if`. The kaggle package is tiny,
# so installing it unconditionally costs nothing and avoids depending on IPython
# transforming a magic inside an indented block.
%pip install -q kaggle

import os
import json

KAGGLE_SLUG = "osamahosamabdellatif/high-quality-invoice-images-for-ocr"
KAGGLE_TOKEN = "/root/.kaggle/kaggle.json"

if USE_KAGGLE and not os.path.exists(KAGGLE_TOKEN):
    from google.colab import files

    print("Upload kaggle.json")
    print("Kaggle -> your avatar -> Settings -> API -> Create New API Token")
    files.upload()
    os.makedirs(os.path.dirname(KAGGLE_TOKEN), exist_ok=True)
    os.replace("kaggle.json", KAGGLE_TOKEN)
    os.chmod(KAGGLE_TOKEN, 0o600)
    print("token installed")
else:
    print("USE_KAGGLE is False - training on generated invoices only.")
    print("That is the current, deliberate state: the Kaggle download carries no labels.")


In [ ]:
if USE_KAGGLE:
    !kaggle datasets download -d $KAGGLE_SLUG -p kaggle_data --unzip -q
    print("downloaded")
else:
    print("USE_KAGGLE is False - skipping the download")

### Look before mapping

This prints the directory tree and one annotation, in full, **before** anything is mapped.
The field names in someone else's dataset are not knowable in advance, and a label mapping
written from a guess fails silently - it trains, the loss falls, and the model learns
nothing useful.

Read the output, then set `FIELD_MAP` in the next cell to match what is actually there.

In [ ]:
import json, os
from collections import Counter
from pathlib import Path

# Always bound, so the converter below can reference it whether or not the
# download ran.
annotation_files = []

if USE_KAGGLE:
    root = Path("kaggle_data")

    # Count every extension. The previous version looked only for .json and .txt and
    # reported '0 json files' while saying nothing about the 8,181 other files - an
    # inspector that only looks for what it expects is not an inspector.
    extensions = Counter(p.suffix.lower() for p in root.rglob('*') if p.is_file())
    print("file types present:")
    for suffix, count in extensions.most_common():
        print(f"  {suffix or '(no extension)':16} {count}")
    print()

    for path in sorted(root.rglob("*"))[:15]:
        marker = "/" if path.is_dir() else f"  ({path.stat().st_size // 1024} KB)"
        print(f"  {path.relative_to(root)}{marker}")

    # Anything that might carry annotations, whatever it is called.
    annotation_files = [
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in {'.json', '.csv', '.txt', '.xlsx', '.jsonl', '.xml'}
    ]
    print()
    print(f"possible annotation files: {len(annotation_files)}")
    if annotation_files:
        first = annotation_files[0]
        print(f"--- {first.name} ---")
        print(first.read_text(encoding="utf-8", errors="ignore")[:2000])
    else:
        print()
        print("NO ANNOTATIONS IN THIS DOWNLOAD - images only.")
        print("A tagger cannot train on unlabelled pictures. Either the labels live")
        print("somewhere else, or this dataset is not usable here. Do not spend time")
        print("on FIELD_MAP until this line changes.")

### Map their field names onto ours

Adjust `FIELD_MAP` to match what the cell above actually printed. Dotted paths walk into
nested objects; a `[]` segment means "a list, take each entry".

`LINE_FIELDS` maps the keys inside one line item. Leave an entry out to ignore that field.

In [ ]:
# EDIT THIS to match the printed annotation. Left side: their path. Right side: our label.
FIELD_MAP = {
    "invoice_number":       "INVOICE_NUMBER",
    "invoice_date":         "ISSUE_DATE",
    "due_date":             "DUE_DATE",
    "seller.name":          "VENDOR_NAME",
    "client.name":          "BUYER_NAME",
    "currency":             "CURRENCY",
    "subtotal":             "SUBTOTAL",
    "tax":                  "TAX",
    "total":                "TOTAL",
}

LINE_ITEMS_PATH = "items"
LINE_FIELDS = {
    "description": "LINE_DESCRIPTION",
    "quantity":    "LINE_QUANTITY",
    "unit_price":  "LINE_UNIT_PRICE",
    "total_price": "LINE_AMOUNT",
}

# Money and totals appear more than once - a grand total is often also a line amount. Header
# money fields are matched from the END of the document, where the totals block sits.
PREFER_LAST = {"SUBTOTAL", "TAX", "TOTAL"}

OCR_TEXT_KEY = None   # set to a key name if the OCR text lives inside the JSON;
                      # leave None if it is a sibling .txt file


In [ ]:
import re

def walk(obj, path):
    """Follow a dotted path into nested dicts. Returns None if any step is missing."""
    for part in path.split("."):
        if not isinstance(obj, dict) or part not in obj:
            return None
        obj = obj[part]
    return obj


def normalise(token):
    """Compare loosely enough to survive formatting differences.

    The annotation may say 1234.56 where the document prints $1,234.56. Comparing raw
    strings would fail on every money field, which would look like a bad mapping.
    """
    token = token.strip().lower().strip(".,:;()")
    if any(ch.isdigit() for ch in token):
        token = re.sub(r"[^0-9.-]", "", token).rstrip(".")
        token = token.lstrip("0") or "0" if token.replace(".", "").isdigit() else token
    return token


def find_span(words, value, prefer_last=False):
    """Locate a value inside the document words.

    Returns every match, not just one. A value often appears more than once - a grand
    total restated at the bottom, or a line whose unit price equals its amount - and the
    caller needs somewhere to fall through to when its first choice is already taken.
    """
    target = [normalise(t) for t in str(value).split() if normalise(t)]
    if not target:
        return []
    normalised = [normalise(w) for w in words]
    hits = [
        i for i in range(len(normalised) - len(target) + 1)
        if normalised[i:i + len(target)] == target
    ]
    if prefer_last:
        hits.reverse()
    return [(start, len(target)) for start in hits]

### Convert, and report how much of it actually aligned

This is the cell that matters. For the generated invoices, labels are attached as the text
is written and are correct by construction. For someone else's data there is no choice but
to **match values back into the text**, and that is exactly where silent labelling bugs
live - a value that cannot be found is a token left tagged `O`, which teaches the model that
the field is not there.

So the converter reports its alignment rate per field rather than trusting itself. Read it:

- **Near 0% on a field** means `FIELD_MAP` names a key that does not exist. Fix the mapping.
- **Near 0% everywhere** means the OCR text is not being found at all.
- **Anything below about 80% on a required field** is worth looking at before training on it.

Documents where a required field could not be aligned are **dropped**, not included with a
missing label. A wrong label is worse than one fewer example.

In [ ]:
from collections import Counter

def convert_document(annotation, text):
    """Return (words, labels), or None if a required field could not be aligned."""
    words = text.split()
    if not words:
        return None, Counter()

    labels = ["O"] * len(words)
    aligned = Counter()

    def tag(value, label):
        if value in (None, "", []):
            return False
        for start, length in find_span(words, value, prefer_last=label in PREFER_LAST):
            if any(labels[i] != "O" for i in range(start, start + length)):
                continue          # taken by another field; try the next occurrence
            for offset in range(length):
                labels[start + offset] = ("B-" if offset == 0 else "I-") + label
            aligned[label] += 1
            return True
        return False

    for path, label in FIELD_MAP.items():
        tag(walk(annotation, path), label)

    for item in (walk(annotation, LINE_ITEMS_PATH) or []):
        if not isinstance(item, dict):
            continue
        for key, label in LINE_FIELDS.items():
            tag(item.get(key), label)

    required_here = ["INVOICE_NUMBER", "VENDOR_NAME", "TOTAL", "CURRENCY"]
    if any(aligned[label] == 0 for label in required_here):
        return None, aligned

    return (words, labels), aligned


In [ ]:
real_examples = []
alignment = Counter()
attempted = dropped = 0

if USE_KAGGLE:
    for json_path in annotation_files:
        try:
            annotation = json.loads(json_path.read_text(encoding="utf-8"))
        except Exception:
            continue

        if OCR_TEXT_KEY:
            text = walk(annotation, OCR_TEXT_KEY) or ""
        else:
            txt_path = json_path.with_suffix(".txt")
            text = (
                txt_path.read_text(encoding="utf-8", errors="ignore")
                if txt_path.exists() else ""
            )

        if not text.strip():
            continue

        attempted += 1
        converted, counts = convert_document(annotation, text)
        alignment.update({k: 1 for k in counts if counts[k]})
        if converted is None:
            dropped += 1
        else:
            real_examples.append(converted)

    print(f"attempted {attempted}, kept {len(real_examples)}, dropped {dropped}")
    if attempted:
        print()
        print("alignment rate per field:")
        for label in FIELD_LABELS:
            rate = alignment[label] / attempted
            flag = "  <-- check FIELD_MAP" if rate < 0.5 else ""
            print(f"  {label:20} {rate:6.1%}{flag}")
    if attempted and not real_examples:
        print()
        print("Nothing aligned. The mapping or the OCR text path is wrong -")
        print("do not train on this until the rates above look sane.")
else:
    print("USE_KAGGLE is False - generated invoices only")

all_examples = examples + real_examples
print()
print(f"{len(all_examples)} training documents total, {len(real_examples)} real")

## 2. Tokenize and align

A word can become several word-pieces. The label goes on the first piece, and the rest get
-100 so the loss ignores them. Getting this wrong is the classic silent bug in token
classification: it trains, the loss falls, and the model is learning the wrong thing.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def encode(batch):
    encoded = tokenizer(
        batch["words"], is_split_into_words=True,
        truncation=True, max_length=512, padding=False,
    )
    all_labels = []
    for i, labels in enumerate(batch["labels"]):
        word_ids = encoded.word_ids(batch_index=i)
        aligned, previous = [], None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)                      # special token
            elif word_id != previous:
                aligned.append(LABEL_TO_ID[labels[word_id]])
            else:
                # Continuation piece. Labelled, NOT masked.
                #
                # Masking these with -100 leaves the model unsupervised on them, so at
                # serving time it predicts O for a continuation and the span breaks -
                # INV-2026-0042 comes back as 'inv'. Evaluation used the same mask and
                # so never saw the problem: it scored first pieces and reported 1.000.
                #
                # A B- label continues as I-; an O stays O.
                label = labels[word_id]
                if label.startswith("B-"):
                    label = "I-" + label[2:]
                aligned.append(LABEL_TO_ID[label])
            previous = word_id
        all_labels.append(aligned)
    encoded["labels"] = all_labels
    return encoded


dataset = Dataset.from_dict({
    "words": [w for w, _ in all_examples],
    "labels": [l for _, l in all_examples],
})
split = dataset.train_test_split(test_size=0.15, seed=20260901)
tokenized = split.map(encode, batched=True, remove_columns=["words", "labels"])
print(tokenized)

## 3. Train

DistilBERT, six epochs. Small enough to fine-tune on a free T4 in a few minutes and small
enough to serve on a CPU afterwards, which matters because the pipeline runs in a container
with no GPU.

In [ ]:
import inspect
import numpy as np
from transformers import (AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          Trainer, TrainingArguments)
from seqeval.metrics import classification_report, f1_score

# Seed everything before the weights are initialised.
#
# Two runs of identical code, identical data and identical epochs produced SUBTOTAL 73.0%
# and 49.0% on the shifted set - a 24-point swing from nothing but training nondeterminism.
# TrainingArguments already defaults to seed=42 and Trainer calls set_seed with it, but that
# happens after this model is constructed, so the classifier head was being initialised from
# whatever state the process happened to be in.
#
# This does not buy full determinism - cuDNN kernel selection and GPU reduction order are
# still free, and a different Colab GPU changes the answer again. It removes one source. The
# honest handling of the rest is to repeat a configuration and quote the range, which is why
# the manifest now records the GPU.
from transformers import set_seed

SEED = 20260901
set_seed(SEED)

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id=LABEL_TO_ID,
)


def compute_metrics(evaluation):
    logits, labels = evaluation
    predictions = np.argmax(logits, axis=-1)
    true, predicted = [], []
    for prediction_row, label_row in zip(predictions, labels):
        true.append([LABELS[l] for l in label_row if l != -100])
        predicted.append(
            [LABELS[p] for p, l in zip(prediction_row, label_row) if l != -100]
        )
    return {"f1": f1_score(true, predicted)}


# transformers renamed evaluation_strategy to eval_strategy. Ask the signature rather
# than pinning a version, so this notebook keeps working as Colab's images move on.
_accepted = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = "eval_strategy" if "eval_strategy" in _accepted else "evaluation_strategy"

training_arguments = TrainingArguments(**{
    "output_dir": "out",
    "seed": SEED,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE * 2,
    "num_train_epochs": EPOCHS,
    "save_strategy": "no",
    "logging_steps": 50,
    "report_to": [],
    _eval_key: "epoch",
})

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

## 4. Per-field results

The overall F1 is the least useful number here. What matters is the per-field breakdown -
which fields it gets right and which it does not - because that is what tells you where the
pipeline will actually send documents to review, and it is what the README should carry.

Expect `LINE_*` fields to be the weak ones. They are the hardest and they are also where
most of the value is.

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score

output = trainer.predict(tokenized["test"])
predictions = np.argmax(output.predictions, axis=-1)

true, predicted = [], []
for prediction_row, label_row in zip(predictions, output.label_ids):
    true.append([LABELS[l] for l in label_row if l != -100])
    predicted.append([LABELS[p] for p, l in zip(prediction_row, label_row) if l != -100])

# Kept in variables, not just printed: the export cell writes them into the model directory
# so the weights carry their own scores. A model whose numbers live only in a notebook
# output someone closed is a model with no numbers.
report_text = classification_report(true, predicted, digits=3)
per_field = classification_report(true, predicted, digits=3, output_dict=True)
overall_f1 = float(f1_score(true, predicted))

print(report_text)
print(f"overall entity F1: {overall_f1:.3f}")


### The number that actually counts: the serving path

The per-field table above scores the model the way it was trained - one prediction per
word-piece. That is not what the pipeline does. At serving time the pieces are
reassembled into spans and mailman reads whole field values, and a metric that never
exercises that step can report a perfect score on a model that returns `inv` for an
invoice number.

So this cell runs the **real serving path** - the aggregation pipeline, exactly as
`mailman/trained.py` calls it - over held-out documents, and compares the reassembled
field values against the gold ones. Quote this number, not the one above.

In [ ]:
from transformers import pipeline as hf_pipeline
from collections import defaultdict

serving = hf_pipeline(
    "token-classification", model=model, tokenizer=tokenizer,
    # Must match mailman/trained.py exactly, or this measures something else.
    aggregation_strategy="first",
    device=0 if model.device.type == "cuda" else -1,
)

held_out = split["test"]
sample_size = min(EVAL_DOCUMENTS, len(held_out))

correct = defaultdict(int)
seen = defaultdict(int)

for row in range(sample_size):
    words, labels = held_out[row]["words"], held_out[row]["labels"]

    gold = defaultdict(list)
    for word, label in zip(words, labels):
        if label != "O":
            gold[label[2:]].append(word)

    predicted = defaultdict(list)
    for span in serving(" ".join(words)):
        predicted[span["entity_group"]].append(span["word"])

    for field in FIELD_LABELS:
        if not gold[field]:
            continue
        seen[field] += 1
        want = " ".join(gold[field]).lower().replace(" ", "")
        got = " ".join(predicted[field]).lower().replace(" ", "").replace("##", "")
        if want == got:
            correct[field] += 1

print(f"exact-match accuracy on the serving path, {sample_size} held-out documents")
print()
for field in FIELD_LABELS:
    if seen[field]:
        rate = correct[field] / seen[field]
        print(f"  {field:20} {rate:6.1%}  ({correct[field]}/{seen[field]})")
print()
total_seen = sum(seen.values())
total_ok = sum(correct.values())
print(f"  {"OVERALL":20} {total_ok / total_seen:6.1%}  ({total_ok}/{total_seen})")
print()
print("If this is far below the per-field F1 above, the difference is the reassembly")
print("step - and this number is the honest one.")

## 4b. A held-out set the generator never produced

The score above is close to meaningless on its own. Training and validation are drawn
from the same generator - the same six vendors, the same eight goods, the same label
wording - so a model that memorises the generator scores perfectly without having
learned anything about invoices.

This cell builds a **shifted** evaluation set: a second generator with no vendor, no
buyer, no product and no label phrasing in common with the training data, plus date and
money formats the model has not seen. Same document type, nothing else shared.

**The gap between the two numbers is the result.** A small gap means the model learned
the structure of an invoice. A large gap means it learned this notebook.

In [ ]:
# Every string here must appear nowhere in the training vocabulary. The assertion at
# the bottom enforces that rather than trusting it - an overlap makes this set
# in-distribution and the whole measurement worthless, and it happened once already.
#
# THIS GENERATOR IS FROZEN. Runs 1 to 4 were scored against it, and changing it would make
# the run table meaningless. Everything that varies belongs in the training generator.
SHIFT_VENDORS = [
    "Zaltman Hydraulics", "Fitzgerald Cold Storage", "Pemberton Acoustics",
    "Nakamura Optics GmbH", "Threlfall Bindery", "Costa Verde Shipping",
]
SHIFT_BUYERS = [
    "Wexley Aggregates", "Duchamp Galleries", "Ingleby Pharmacy Group",
    "Saltburn Leisure", "Montrose Chandlers",
]
SHIFT_GOODS = [
    "Hydraulic hose replacement", "Cold chain monitoring", "Acoustic panel fitting",
    "Lens polishing service", "Perfect binding, per 100", "Container demurrage",
    "Night shift premium", "Dilapidations survey",
]

SHIFT_NUMBER_LABELS = [
    "Advice note", "Charge slip ID", "Docket", "Voucher number",
]
SHIFT_DATE_LABELS = [
    "Rendered", "Supply date", "Executed on", "Period ending",
]
SHIFT_DUE_LABELS = [
    "Discharge by", "Clearance required", "Funds required by",
]
SHIFT_SUBTOTAL_LABELS = [
    "Chargeable value", "Pre-levy amount", "Aggregate of items",
]
SHIFT_TAX_LABELS = [
    "Levy", "Impost", "Statutory charge",
]
SHIFT_TOTAL_LABELS = [
    "Settlement figure", "Remittance required", "Sum owing", "Final charge",
]
SHIFT_HEADERS = [
    "Narrative Count Tariff Extension", "Line Volume Charge Subtotal",
]
SHIFT_CURRENCY_LABELS = [
    "Denominated in", "Invoice currency is",
]


def generate_shifted(rng):
    """Same document type as the training generator, sharing none of its wording."""
    words, labels = [], []

    def emit(text, label=None):
        parts = str(text).split()
        for i, part in enumerate(parts):
            words.append(part)
            labels.append("O" if label is None else ("B-" if i == 0 else "I-") + label)

    code_, symbol = rng.choice(CURRENCIES)
    issued = date(2026, 1, 1) + timedelta(days=rng.randrange(360))
    due = issued + timedelta(days=rng.choice([7, 21, 90]))
    style = rng.randrange(DATE_STYLES)

    emit(rng.choice(SHIFT_VENDORS), "VENDOR_NAME")
    emit("Registered office, Leeds")
    emit(rng.choice(SHIFT_NUMBER_LABELS))
    emit(f"{rng.randrange(100, 999)}/{issued.year}/{rng.randrange(10, 99)}", "INVOICE_NUMBER")
    emit(rng.choice(SHIFT_DATE_LABELS))
    emit(a_date(issued, style), "ISSUE_DATE")
    emit(rng.choice(SHIFT_DUE_LABELS))
    emit(a_date(due, style), "DUE_DATE")
    emit("Account of")
    emit(rng.choice(SHIFT_BUYERS), "BUYER_NAME")
    emit(rng.choice(SHIFT_HEADERS))

    subtotal = 0.0
    for _ in range(rng.randrange(1, 6)):
        qty = rng.randrange(1, 20)
        unit = round(rng.uniform(9, 700), 2)
        amount = round(qty * unit, 2)
        subtotal += amount
        emit(rng.choice(SHIFT_GOODS), "LINE_DESCRIPTION")
        emit(str(qty), "LINE_QUANTITY")
        emit(money(unit, symbol), "LINE_UNIT_PRICE")
        emit(money(amount, symbol), "LINE_AMOUNT")

    subtotal = round(subtotal, 2)
    tax = round(subtotal * rng.choice([0.0, 0.175, 0.2]), 2)
    emit(rng.choice(SHIFT_SUBTOTAL_LABELS))
    emit(money(subtotal, symbol), "SUBTOTAL")
    emit(rng.choice(SHIFT_TAX_LABELS))
    emit(money(tax, symbol), "TAX")
    emit(rng.choice(SHIFT_TOTAL_LABELS))
    emit(money(round(subtotal + tax, 2), symbol), "TOTAL")
    emit(rng.choice(SHIFT_CURRENCY_LABELS))
    emit(code_, "CURRENCY")

    return words, labels


# Enforced, not hoped for. Every training list, including the ones added for run 5.
_training_phrases = set()
for _phrases in VOCABULARY.values():
    _training_phrases.update(_phrases)

_shift_phrases = set(
    SHIFT_VENDORS + SHIFT_BUYERS + SHIFT_GOODS + SHIFT_NUMBER_LABELS
    + SHIFT_DATE_LABELS + SHIFT_DUE_LABELS + SHIFT_SUBTOTAL_LABELS
    + SHIFT_TAX_LABELS + SHIFT_TOTAL_LABELS + SHIFT_HEADERS + SHIFT_CURRENCY_LABELS
)
_shared = _training_phrases & _shift_phrases
assert not _shared, f"shifted set shares phrases with training: {sorted(_shared)}"

# Composed descriptions are new in run 5 and are not in any list, so the phrase check above
# cannot see them. Every combination the generator can produce, against the shifted goods.
_composed = {
    f"{p}{g}{s}"
    for g in GOODS for p in GOODS_PREFIXES for s in GOODS_SUFFIXES
}
assert not (_composed & set(SHIFT_GOODS)), "a composed description collides with SHIFT_GOODS"

# Invoice-number shapes are new in run 5 too. The shifted set writes \d{3}/\d{4}/\d{2}; if
# the training generator can produce that shape, INVOICE_NUMBER stops being held out and the
# field this run is trying to fix becomes the one field it cannot measure.
import re as _re

_probe = random.Random(4242)
_shift_shape = _re.compile(r"^\d{3}/\d{4}/\d{2}$")
_collisions = {
    n for n in (an_invoice_number(_probe, 2026) for _ in range(20000))
    if _shift_shape.match(n)
}
assert not _collisions, f"training invoice numbers collide with the shifted shape: {sorted(_collisions)[:5]}"

shift_rng = random.Random(77771)
shifted = [generate_shifted(shift_rng) for _ in range(300)]

overlap = set(w for ws, _ in shifted for w in ws) & set(w for ws, _ in examples for w in ws)
print(f"{len(shifted)} shifted documents, no shared phrases")
print(f"composed descriptions checked: {len(_composed)}, none collide with the shifted goods")
print(f"invoice-number shapes checked: 20000 draws, none match the shifted shape")
print(f"token-level overlap with training: {len(overlap)} tokens")
print("(digits, currency codes and punctuation overlap by necessity; wording does not)")


In [ ]:
from collections import defaultdict

# Per-field shifted scores from run 5. Run 5 removed the remaining constants - one invoice
# number shape, one field order, twenty-two descriptions - and took INVOICE_NUMBER 13->99%,
# ISSUE_DATE 12.5->68%, TOTAL 80.5->99.5%, overall 70.7->84.1%.
#
# What it left behind is two PAIRS sitting on identical numbers, which is not a coincidence:
#
#     ISSUE_DATE 68.0% (136/200)    DUE_DATE 68.0% (136/200)
#     SUBTOTAL   73.0% (146/200)    TAX      73.0% (146/200)
#
# Two adjacent fields of the same type, scoring identically, is what within-pair confusion
# looks like: when it gets one right it gets both right. DUE_DATE also REGRESSED 24.5 points,
# and the explanation is that run 4 was reading it positionally - training always put the
# issue date before the due date, and so does the shifted set, so position transferred for
# free. Shuffling the field order took that crutch away and exposed that the model cannot
# tell the two dates apart from an unseen label word.
BASELINE_PER_FIELD = {
    "INVOICE_NUMBER": 0.990, "VENDOR_NAME": 0.995, "BUYER_NAME": 0.690,
    "ISSUE_DATE": 0.680, "DUE_DATE": 0.680, "CURRENCY": 1.000,
    "SUBTOTAL": 0.730, "TAX": 0.730, "TOTAL": 0.995,
    "LINE_DESCRIPTION": 0.460, "LINE_QUANTITY": 0.985,
    "LINE_UNIT_PRICE": 1.000, "LINE_AMOUNT": 1.000,
}
WATCH = {"ISSUE_DATE", "DUE_DATE", "SUBTOTAL", "TAX", "LINE_DESCRIPTION"}   # the pairs, and descriptions


def serving_accuracy(pairs, label):
    """Exact-match accuracy per field, through the real aggregation pipeline."""
    correct, seen = defaultdict(int), defaultdict(int)

    for words, labels in pairs:
        gold = defaultdict(list)
        for word, tag in zip(words, labels):
            if tag != "O":
                gold[tag[2:]].append(word)

        predicted = defaultdict(list)
        for span in serving(" ".join(words)):
            predicted[span["entity_group"]].append(span["word"])

        for field in FIELD_LABELS:
            if not gold[field]:
                continue
            seen[field] += 1
            want = "".join(gold[field]).lower().replace(" ", "")
            got = "".join(predicted[field]).lower().replace(" ", "").replace("##", "")
            if want == got:
                correct[field] += 1

    print(f"--- {label} ---")
    rates = {}
    for field in FIELD_LABELS:
        if seen[field]:
            rate = correct[field] / seen[field]
            rates[field] = rate
            print(f"  {field:20} {rate:6.1%}  ({correct[field]}/{seen[field]})")
    total_seen, total_ok = sum(seen.values()), sum(correct.values())
    overall = total_ok / total_seen if total_seen else 0.0
    print(f"  {'OVERALL':20} {overall:6.1%}  ({total_ok}/{total_seen})")
    print()
    return overall, rates


in_distribution = [
    (all_examples[i][0], all_examples[i][1])
    for i in range(min(EVAL_DOCUMENTS, len(all_examples)))
]

same, _ = serving_accuracy(in_distribution, "SAME generator as training")
shift, shift_rates = serving_accuracy(shifted[:EVAL_DOCUMENTS], "SHIFTED - vocabulary and labels never seen")

print("=" * 70)
print(f"in-distribution {same:.1%}   shifted {shift:.1%}   gap {same - shift:.1%}")
print(f"run {BASELINE['run']} baseline                   shifted {BASELINE['shifted']:.1%}   gap {BASELINE['gap']:.1%}")
print(f"movement                         {shift - BASELINE['shifted']:+.1%} shifted, "
      f"{(same - shift) - BASELINE['gap']:+.1%} gap")
print("=" * 70)

if EPOCHS != BASELINE["epochs"]:
    print()
    print(f"NOT A VALID COMPARISON: {EPOCHS} epochs against the baseline's "
          f"{BASELINE['epochs']}. Two variables moved.")

print()
print(f"Per field, against run {BASELINE['run']}. The fields this run targets are marked:")
print("two pairs on identical numbers (issue/due, subtotal/tax) and the descriptions.")
print()
print(f"  {'FIELD':20} {'run ' + str(BASELINE['run']):>8} {'now':>8} {'move':>8}")
for field in FIELD_LABELS:
    now = shift_rates.get(field)
    before = BASELINE_PER_FIELD.get(field)
    if now is None or before is None:
        continue
    marker = "  <-- targeted by this run" if field in WATCH else ""
    print(f"  {field:20} {before:8.1%} {now:8.1%} {now - before:+8.1%}{marker}")

print()
print("How to read it:")
print("  The three marked fields move  -> removing the remaining constants worked, and")
print("                                   the ablations are worth running to say which")
print("                                   of the three did it.")
print("  They stay flat                -> read the diagnostic cell below before doing")
print("                                   anything. If the wanted value is being swallowed")
print("                                   by a neighbouring span, more variety in THIS")
print("                                   field is the wrong fix - the boundary is.")
print("  Anything else regresses       -> per-field noise was measured at 6 points over")
print("                                   100 documents; this runs 200, so treat a move")
print("                                   under about 5 points as nothing.")


In [ ]:
# Wanted versus got, on the fields the run got wrong. Read this before changing anything.
#
# A per-field percentage says a field is wrong; it does not say HOW. Twice now the "how" has
# been the whole finding. In run 2 this print revealed that adjacent fields were merging into
# one span - the invoice number, issue date and due date arriving as a single ISSUE_DATE -
# which reframed the problem from "it has not learned the vocabulary" to "it cannot find a
# boundary", and run 4 confirmed that by fixing the neighbours' labels and watching TOTAL
# gain 74 points without its own vocabulary changing.
#
# So: diagnose, then fix. Never the other way round.
WORST_N = 4
SAMPLES = 6

_rates = sorted(shift_rates.items(), key=lambda kv: kv[1])[:WORST_N]
print(f"The {WORST_N} weakest fields on the shifted set:")
for field, rate in _rates:
    print(f"  {field:20} {rate:6.1%}")
print()
print("=" * 78)

for field, rate in _rates:
    print()
    print(f"### {field}  ({rate:.1%})")
    print()
    shown = 0
    for words, labels in shifted[:200]:
        gold = [w for w, t in zip(words, labels) if t.endswith(field)]
        if not gold:
            continue
        predicted = defaultdict(list)
        for span in serving(" ".join(words)):
            predicted[span["entity_group"]].append(span["word"])
        want = "".join(gold).lower().replace(" ", "")
        got = "".join(predicted[field]).lower().replace(" ", "").replace("##", "")
        if want == got:
            continue
        print(f"  wanted  {want}")
        print(f"  got     {got or '(nothing)'}")
        # If the wanted string turns up INSIDE another field's span, this is a merge and not
        # a misread, and the fix belongs on the neighbouring field's labels rather than here.
        for other, tokens in predicted.items():
            if other == field:
                continue
            blob = "".join(tokens).lower().replace(" ", "").replace("##", "")
            if want and want in blob:
                print(f"          ^ swallowed by {other}: {blob[:90]}")
        print()
        shown += 1
        if shown >= SAMPLES:
            break

print("=" * 78)
print("Reading it:")
print("  got is EMPTY, and wanted appears inside another field  -> spans are merging.")
print("     Fix the boundary, which means the neighbouring labels or values, not this field.")
print("  got is a PREFIX or fragment of wanted                  -> reassembly, not tagging.")
print("  got is a different, plausible value from the document  -> genuinely mislabelled.")
print("     That is the only case where more examples of THIS field is the right answer.")


## 5. Try it on one document

The same shape the pipeline will see: raw text in, tagged spans out.

In [ ]:
from transformers import pipeline as hf_pipeline

tagger = hf_pipeline("token-classification", model=model, tokenizer=tokenizer,
                     aggregation_strategy="simple",
                     device=0 if model.device.type == "cuda" else -1)

sample = """Northgate Supplies
82 Mill Road
INVOICE
Invoice No. INV-2026-7741
Date of Issue: 04 March 2026
Due: 18/03/2026
Bill To: Orchard Foods
Description Qty Unit Price Amount
Consulting hours 12 GBP 85.00 GBP 1,020.00
Subtotal GBP 1,020.00
VAT GBP 204.00
Grand Total GBP 1,224.00
Currency GBP"""

for span in tagger(sample):
    print(f"{span['entity_group']:18} {span['score']:.3f}  {span['word']}")


## 6. Export

Everything the pipeline needs, written to `models/extractor/`:

- the weights and config,
- the tokenizer,
- `mailman_model.json` - the label set, the base model, the training set size, and the
  scores from section 4.

That last file is the point of this cell. A set of weights whose numbers live only in a
notebook output someone closed is a set of weights with no numbers. It travels with the
model, `mailman/trained.py` reads it on load, and it is what gets pasted into `NOTES.md`.

In [ ]:
import json, os, platform, shutil, sys
import torch
from datetime import datetime, timezone

EXPORT_DIR = "models/extractor"
os.makedirs(EXPORT_DIR, exist_ok=True)

model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

manifest = {
    "name": "mailman-extractor",
    "task": "token-classification",
    "base_model": BASE_MODEL,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "labels": LABELS,
    "field_labels": FIELD_LABELS,
    "training_examples": len(tokenized["train"]),
    "eval_examples": len(tokenized["test"]),
    "real_examples": len(real_examples),
    "generated_only": len(real_examples) == 0,
    "epochs": int(trainer.args.num_train_epochs),
    "seed": int(trainer.args.seed),
    # Recorded because two runs of the same configuration differ by up to 24 points on a
    # field, and a different GPU is one of the reasons. Without this, a run cannot be
    # compared to another run at all.
    "gpu": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    "overall_entity_f1": round(overall_f1, 4),
    # The token-level F1 above is measured on the training generator's own
    # split and is close to meaningless alone. These two are the honest ones.
    "serving_in_distribution": round(same, 4),
    "serving_shifted": round(shift, 4),
    "generalisation_gap": round(same - shift, 4),
    # What this run is a comparison against, and whether it was a fair one.
    "baseline": BASELINE,
    "comparable_to_baseline": int(trainer.args.num_train_epochs) == BASELINE["epochs"],
    # How many distinct phrases from each vocabulary list the generator actually emitted.
    #
    # Recorded because four of these lists were defined and never drawn from for three
    # consecutive runs, and nothing in the manifest said so - the weights carried a
    # "greatly enlarged vocabulary" note describing a change that had half happened, and
    # SUBTOTAL and TAX sat at 0% because of it. A run's data is now described by the run.
    "vocabulary_phrases_emitted": {
        name: [len(DRAWN_PHRASES[name]), len(phrases)]
        for name, phrases in sorted(VOCABULARY.items())
    },
    "date_formats": DATE_STYLES,
    # Which kinds of variation were switched on. Without this a set of weights cannot
    # say which experiment produced it, which is how three runs came to be described
    # as "enlarged vocabulary" when four of the lists were never drawn from.
    "generator": dict(GENERATOR),
    "per_field": {
        key: {metric: round(float(value), 4) for metric, value in scores.items()}
        for key, scores in per_field.items()
        if isinstance(scores, dict)
    },
    "per_field_shifted": {
        field: round(rate, 4) for field, rate in sorted(shift_rates.items())
    },
    "max_sequence_length": 512,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "caveats": [
        "Trained on generated invoices." if not real_examples else
        f"Trained on generated invoices plus {len(real_examples)} real ones.",
        "Text only - the model never sees the page, so column layout is invisible to it.",
        "Truncates at 512 word-pieces; a long multi-page invoice is silently cut short.",
        "Field order is identical in every training document, so position is a perfect cue "
        "in training and none at all on a real invoice. This is the known structural limit.",
    ],
}

if not manifest["comparable_to_baseline"]:
    manifest["caveats"].append(
        f"Trained for {manifest['epochs']} epochs against the baseline's "
        f"{BASELINE['epochs']}: the shifted score is NOT a one-variable comparison."
    )

with open(os.path.join(EXPORT_DIR, "mailman_model.json"), "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(json.dumps(
    {k: v for k, v in manifest.items() if k not in ("per_field", "labels")},
    indent=2,
))


### Verify before downloading

The export is reloaded from disk and run on a document *before* the zip is made. A 250 MB
download that turns out to be missing a tokenizer file is 250 MB of wasted time and a
confusing failure on the other machine. Checking here costs seconds.

In [ ]:
import os

# What from_pretrained actually needs. Either tokenizer flavour is fine: a fast
# tokenizer writes tokenizer.json and may not write vocab.txt at all, and recent
# transformers folds special_tokens_map.json into tokenizer_config.json.
REQUIRED_ANY = {
    "config":           ["config.json"],
    "weights":          ["model.safetensors", "pytorch_model.bin"],
    "tokenizer":        ["tokenizer.json", "vocab.txt"],
    "tokenizer config": ["tokenizer_config.json"],
    "manifest":         ["mailman_model.json"],
}

present = set(os.listdir(EXPORT_DIR))
print("files:", ", ".join(sorted(present)))
print()

missing = [name for name, options in REQUIRED_ANY.items() if not (present & set(options))]
if missing:
    print(f"possibly incomplete: {missing}")
    print("the reload below is the real test - if it passes, the export is fine")

# The check that actually matters. Load it back from disk on CPU, the way the container
# will, and see whether it still works. File names are a proxy; this is the truth.
reloaded = hf_pipeline("token-classification", model=EXPORT_DIR,
                       aggregation_strategy="simple", device=-1)
spans = reloaded(sample)
assert spans, "the reloaded model tagged nothing - do not ship this"

print(f"reloaded from disk on CPU and tagged {len(spans)} spans - export is good")

### Zip and download

The download starts on its own. If your browser blocks it, or the connection drops partway
through a 250 MB transfer, the Drive fallback in the next cell is the reliable route.

In [ ]:
ZIP_NAME = "mailman-extractor.zip"
shutil.make_archive("mailman-extractor", "zip", EXPORT_DIR)
size_mb = os.path.getsize(ZIP_NAME) / 1_000_000
print(f"{ZIP_NAME}  {size_mb:.0f} MB")

try:
    from google.colab import files
    files.download(ZIP_NAME)          # starts automatically
    print("download started - check your browser")
except ImportError:
    print(f"Not on Colab. The zip is at {os.path.abspath(ZIP_NAME)}")
except Exception as exc:
    print(f"Automatic download failed ({type(exc).__name__}). Use the Drive cell below.")


In [ ]:
# Fallback, and the better option for a 250 MB file on a slow connection: save to Drive
# and download from there, which resumes if it drops. Uncomment to use.
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy(ZIP_NAME, "/content/drive/MyDrive/mailman-extractor.zip")
# print("saved to Drive")


## 7. Everything you need, printed in one place

Run this and copy what it gives you. The commands go in the terminal; the table goes in
`NOTES.md`.

In [ ]:
print("=" * 72)
print("ON THE MACHINE RUNNING MAILMAN (PowerShell, from the mailman folder)")
print("=" * 72)
# Forward slashes throughout: PowerShell and curl.exe both accept them on Windows, and it
# keeps this cell free of backslash escaping that would otherwise have to survive an
# f-string, a notebook file and a copy-paste.
print(f"""
Expand-Archive {ZIP_NAME} -DestinationPath ./models/extractor -Force
$env:MAILMAN_EXTRACTOR = "trained"
docker compose up -d --build

curl.exe -F "file=@corpus/01-clean.pdf" http://localhost:8000/documents
Invoke-RestMethod http://localhost:8000/documents/<id>/extraction | ConvertTo-Json -Depth 10

The extraction row should now read  model_name: trained:extractor
and  prompt_version: {manifest['name']}
""")

print("=" * 72)
print("PASTE INTO NOTES.md")
print("=" * 72)

# The honest numbers come FIRST and the token-level F1 comes last, marked.
#
# This cell used to lead with `Overall F1 1.000` and thirteen per-field rows of 1.000, which
# is the score on the training generator's own split - the number three entries of the log
# call meaningless, because train and test are drawn from the same generator. A cell whose
# entire job is "paste this into NOTES.md" was handing over the one figure that must never be
# quoted alone, and omitting the serving and shifted scores that the manifest already carried.
valid = "yes" if manifest["comparable_to_baseline"] else "NO - see the warning below"

print(f"""
### Trained extractor - {manifest['trained_at'][:10]}

Base model      {manifest['base_model']}
Training set    {manifest['training_examples']} examples """
f"""({'generated only' if manifest['generated_only'] else str(manifest['real_examples']) + ' real'})
Epochs          {manifest['epochs']}

THE NUMBERS THAT MEAN SOMETHING - exact match through the real serving path:

  in-distribution   {manifest['serving_in_distribution']:.1%}
  shifted           {manifest['serving_shifted']:.1%}
  gap               {manifest['generalisation_gap']:.1%}

  run {BASELINE['run']} baseline   shifted {BASELINE['shifted']:.1%}   gap {BASELINE['gap']:.1%}
  movement          {manifest['serving_shifted'] - BASELINE['shifted']:+.1%} shifted, """
f"""{manifest['generalisation_gap'] - BASELINE['gap']:+.1%} gap
  one-variable comparison against the baseline: {valid}

The gap is the result. A small gap means the model learned what an invoice is.
A large gap means it learned this notebook.
""")

if not manifest["comparable_to_baseline"]:
    print(f"  WARNING: {manifest['epochs']} epochs against the baseline's "
          f"{BASELINE['epochs']}. Vocabulary AND training time both moved.")
    print("  Do not record this as a comparison against run 3.")
    print()

print(f"Per field on the shifted set, against run {BASELINE['run']}:")
print()
print(f"  {'FIELD':20} {'run ' + str(BASELINE['run']):>8} {'now':>8} {'move':>8}")
for field in FIELD_LABELS:
    now = manifest["per_field_shifted"].get(field)
    if now is None:
        continue
    before = BASELINE_PER_FIELD.get(field)
    if before is None:
        print(f"  {field:20} {'-':>8} {now:8.1%}")
        continue
    marker = "  <-- targeted by this run" if field in WATCH else ""
    print(f"  {field:20} {before:8.1%} {now:8.1%} {now - before:+8.1%}{marker}")

print()
print("Vocabulary the generator actually emitted (distinct phrases / list size).")
print("Recorded because four of these lists were defined and never drawn from for three")
print("consecutive runs, and no manifest could have contradicted the claim that they were:")
print()
for name, (drawn, size) in manifest["vocabulary_phrases_emitted"].items():
    flag = "" if drawn == size else "   <-- not fully drawn"
    print(f"  {name:20} {drawn:3}/{size:<3}{flag}")
print(f"  {'date formats':20} {manifest['date_formats']}")

print()
print(f"Token-level entity F1 on the in-distribution split: {manifest['overall_entity_f1']:.3f}")
print("  Quote this ONLY with the sentence that follows it. Train and test come from the")
print("  same generator, so it measures memorisation and has been 1.000 on every run,")
print("  including the run that could not find a total on an unfamiliar invoice.")

print()
print("Caveats that must travel with these numbers:")
for caveat in manifest["caveats"]:
    print(f"  - {caveat}")

print("""
Still to do: run the same corpus through the heuristic extractor and record the gap.
The heuristic is what deploys, so a 250 MB model that cannot beat it is not worth shipping,
and reporting that honestly is worth more than reporting a good F1 with no comparison.""")


## What the numbers do and do not say

The per-field table is the useful output; the overall F1 is nearly meaningless on its own.
Expect the `LINE_*` fields to be the weak ones - they are the hardest and they are where
most of the value is.

And the number that actually matters is not in this notebook. It is the **gap** between this
model and the heuristic extractor on the same corpus. A 250 MB model that cannot beat keyword
matching on invoices is not worth shipping, and reporting that honestly is worth more than
reporting a good F1 with no comparison.

## Known limits of this model

- Trained on generated invoices, so it has seen one author's idea of a layout. Mixing in
  SROIE, CORD or FUNSD in the Optional cell is what would change that.
- Text only. It never sees the page, so a two-column layout that interleaves in the text
  layer is invisible to it. LayoutLMv3 with bounding boxes from pdfplumber is the upgrade,
  and it is a real one.
- 512 word-pieces. A long multi-page invoice is truncated, silently.
- It tags spans; it does not understand. It cannot infer a total that was never printed, and
  it should not - the validation rules check arithmetic in Python for exactly that reason.
